In [38]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import RandomizedSearchCV,GridSearchCV
from sklearn.metrics import accuracy_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import warnings

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [2]:
warnings.filterwarnings('ignore')

In [3]:
df=pd.read_csv('data/stud.csv')

In [4]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


LET US DISTINGUISH BETWEEN INDEPENDENT AND DEPENDENT FEATURE

In [6]:
x=df.drop(columns=['math_score'],axis=1)

In [7]:
y=df['math_score']

In [9]:
x.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [12]:
y.head(5)

0    72
1    69
2    90
3    47
4    76
Name: math_score, dtype: int64

In [25]:
cat_features=x.select_dtypes(include="object").columns
num_features=x.select_dtypes(exclude="object").columns

In [26]:
cat_features

Index(['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch',
       'test_preparation_course'],
      dtype='object')

In [27]:
num_features

Index(['reading_score', 'writing_score'], dtype='object')

In [28]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

num_preprocessor=StandardScaler()
cat_preprocessor=OneHotEncoder()

preprocessor=ColumnTransformer(
    [
    ("numerical",num_preprocessor,num_features),
    ("categorical",cat_preprocessor,cat_features)
    ]
)

In [29]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [18]:
x_train.shape


(800, 7)

In [30]:
x_train=preprocessor.fit_transform(x_train)
x_test=preprocessor.transform(x_test)

NOW LET MAKE A FUNCTION TO CALCULATE DIFFERENT METRICS OF FOR MODEL

In [47]:
def calculatemetrics(true,pred):
    mae=mean_absolute_error(true,pred)
    mse=mean_squared_error(true,pred)
    rmse=np.sqrt(mse)
    r2score=r2_score(true,pred)
    return mae,mse,rmse,r2score

In [64]:
models={
    "LinearRegression":LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list=[]
train_r2_list=[]
test_r2_list=[]
diff_list=[]
model_values=list(models.values())
model_keys=list(models.keys())

for i in range(len(list((models)))):
    curr_model=list(model_values)[i]
    curr_model.fit(x_train,y_train)
    

    y_train_pred=curr_model.predict(x_train)
    y_test_pred=curr_model.predict(x_test)

    # Evaluate metrics value for train as well as test data

    model_train_mae , model_train_mse, model_train_rmse, model_train_r2score = calculatemetrics(y_train, y_train_pred)
    model_test_mae , model_test_mse, model_test_rmse, model_test_r2score = calculatemetrics(y_test, y_test_pred)
    model_list.append(model_keys[i])
    print(model_keys[i])
    print("Performance for training data")
    print(f"Mean Absolute Error:- {model_train_mae}")
    print(f"Mean Squared Error:- {model_train_mse}")
    print(f"Root Mean Squared Error:- {model_train_rmse}")
    print(f"R2_Score :- {model_train_r2score}")
    train_r2_list.append(model_test_r2score)
    print("----------------------------")
    print("Performance for testing data")
    print(f"Mean Absolute Error:- {model_test_mae}")
    print(f"Mean Squared Error:- {model_test_mse}")
    print(f"Root Mean Squared Error:- {model_test_rmse}")
    print(f"R2_Score :- {model_test_r2score}")
    print()
    test_r2_list.append(model_test_r2score)
    diff_list.append(model_train_r2score-model_test_r2score)

LinearRegression
Performance for training data
Mean Absolute Error:- 4.266711846071956
Mean Squared Error:- 28.33487038064859
Root Mean Squared Error:- 5.323050852720514
R2_Score :- 0.8743172040139593
----------------------------
Performance for testing data
Mean Absolute Error:- 4.214763142474849
Mean Squared Error:- 29.09516986671547
Root Mean Squared Error:- 5.393993869732841
R2_Score :- 0.8804332983749565

Lasso
Performance for training data
Mean Absolute Error:- 5.205260274468427
Mean Squared Error:- 43.46106018771194
Root Mean Squared Error:- 6.592500298650879
R2_Score :- 0.8072231322208645
----------------------------
Performance for testing data
Mean Absolute Error:- 5.155701094273798
Mean Squared Error:- 42.47556715227398
Root Mean Squared Error:- 6.517328221922997
R2_Score :- 0.8254465092551198

Ridge
Performance for training data
Mean Absolute Error:- 4.265005112727169
Mean Squared Error:- 28.337741791088874
Root Mean Squared Error:- 5.323320560617111
R2_Score :- 0.874304467

In [67]:
pd.DataFrame(zip(model_list, train_r2_list,test_r2_list,diff_list),columns=['Model Name', 'Train_R2_Score','Test_R2_Score','Difference'])

,Model Name,Train_R2_Score,Test_R2_Score,Difference
0,LinearRegression,0.880433,0.880433,-0.006116
1,Lasso,0.825447,0.825447,-0.018223
2,Ridge,0.880592,0.880592,-0.006287
3,K-Neighbors Regressor,0.786127,0.786127,0.069965
4,Decision Tree,0.765039,0.765039,0.234614
5,Random Forest Regressor,0.853105,0.853105,0.123265
6,XGBRegressor,0.821221,0.821221,0.174279
7,CatBoosting Regressor,0.851831,0.851831,0.107105
8,AdaBoost Regressor,0.845157,0.845157,-0.000156
